### INDEXING : CREATING A CHROMA VECOTR STORE

In [10]:
import config
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
import numpy as np

from langchain_chroma import Chroma # It allows developers to store, index, and query vector embeddings (numerical representations of text, images, etc.) to enable semantic search and Retrieval-Augmented Generation (RAG) applications. 
from langchain_community.vectorstores import Chroma 
from langchain_chroma import Chroma  # New way

In [5]:
loader_docx= Docx2txtLoader("../../Data/Introduction_to_Data_and_Data_Science2.docx")
pages = loader_docx.load()

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on =[("#", "Course Title"),
                                                               ("##", "Lecture Title")])

pages_md_splitter = md_splitter.split_text(pages[0].page_content)

for i in range(len(pages_md_splitter)):
    pages_md_splitter[i].page_content = ' '.join(pages_md_splitter[i].page_content.split())

char_splitter = CharacterTextSplitter(separator =".", chunk_size = 500, chunk_overlap = 50)

page_char_spliiter = char_splitter.split_documents(pages_md_splitter)

embedding = OpenAIEmbeddings(model="text-embedding-ada-002",
                            openai_api_key = config.api_key)

In [6]:
len(page_char_spliiter)

20

Now, rather than embedding each document separately as we did in the previous lesson, we'll utilize

a vector store and embed all documents simultaneously.



Lang chain supports integrations with many vector stores, and you can browse them all in the integrations

menu.

The vector store I've chosen for this course is chroma.



In [8]:
#We want this vector store to keep all 20 documents from the page's character split list, and their vector representations.
#This is achieved by invoking the from documents method.
vectorstore = Chroma.from_documents(documents=page_char_spliiter,
                                   embedding= embedding,
                                    persist_directory= './vectore representrtion folder')

#Note that persist directory is not a required parameter.
#If you don't specify it, the vector store will only exist until the kernel is restarted or shut down.

### We'll demonstrate shortly that persisting a vector collection allows us to retrieve it even if we restart the kernel.
### This avoids embedding all tokens anew, which is beneficial since, as you know, embedding is a token consuming process.

### Since the documents and the embeddings are stored locally, we should be able to create a new vector

### store object using the files from this folder.

### Let's do so.



In [11]:
vectorstore_from_dir = Chroma( persist_directory= './vectore representrtion folder',
                                   embedding_function= embedding
                                   )


We also need to pass the embedding object used to create this vector store initially.

This is necessary because the documents in the vector store can be updated or new ones can be added.

So the vector store needs to know which embedding function to use to maintain consistency and accuracy

in the representations of existing and newly added documents.

Additionally, suppose this vector database is used as part of a context aware chatbot application.

In that case, it should be able to embed the user's input with the same embedding function to find

semantically similar documents.



The vector store object was created by applying Chromas from documents method using an embedding function

to embed all documents in a list and specifying a local directory to store the files.

The vector store from directory object was in turn created from an existing database by assigning it

as an instance of the chroma class.

